# DeltaNet v2: Performance Benchmark on Google Colab (Tesla T4)
### Comparing DeltaNet v2 (SM75 CuTe) against NVIDIA GatedDeltaNet-2 and FLA

> **Lưu ý / Container Compatibility**: Notebook được thiết kế tương thích chuẩn với môi trường **Google Colab runtime / container 2026.04 trở về trước** (Python 3.10 / 3.11, PyTorch $\le$ 2.5, C++17). Không cần thực hiện cài đặt hạ cấp hệ thống.

This notebook downloads and executes the benchmark suite from `/content/Deltanet/tests/` to evaluate:
- **Runtime Performance**: Latency (ms) and Throughput (tokens/sec)
- **Peak Memory Consumption**: VRAM allocated & reserved (MB)

**Baselines Compared:**
1. **DeltaNet v2 (Ours)**: Fused CUDA kernels targeting NVIDIA Turing SM75 (Tesla T4, RTX 20-series) with independent $k, v$ gating.
2. **FLA GatedDeltaNet-2**: NVIDIA's official Gated DeltaNet-2 paper architecture implemented via Flash Linear Attention (`fla.layers.GatedDeltaNet2`).
3. **FLA GatedDeltaNet (v1)**: Standard Gated DeltaNet linear attention layer (`fla.layers.GatedDeltaNet`).
4. **PyTorch Reference**: Unfused PyTorch eager baseline.

## 1. Verify GPU Hardware
Confirm that Google Colab is running with an **NVIDIA Tesla T4** GPU (SM75, 16 GB VRAM).

In [ ]:
!nvidia-smi

## 2. Clone Repository & Build DeltaNet v2
Clones `https://github.com/CBG6682/Varlen-Gated-DeltaNet.git` (with dedicated `third_party/cutlass` submodule) and compiles the SM75 fused CUDA extensions using C++17.

**Tùy chọn biên dịch (Build Options)**:
- **Mặc định**: Nếu chưa có file biên dịch `.so`, `setup.py` sẽ tự động biên dịch từ mã nguồn C++/CUDA. Nếu đã có sẵn file `.so` trong `deltanet_v2/`, nó sẽ dùng lại ngay lập tức.
- **Biên dịch lại (Recompile & Overwrite)**: Đặt biến môi trường `FORCE_REBUILD=1` (ví dụ `!FORCE_REBUILD=1 pip install -v . --no-build-isolation`) khi bạn chỉnh sửa mã nguồn và muốn biên dịch đè lại.

In [ ]:
# Clone repository with dedicated third_party/cutlass submodule into /content/Deltanet
!rm -rf /content/Deltanet
!git clone --recursive https://github.com/CBG6682/Varlen-Gated-DeltaNet.git /content/Deltanet
%cd /content/Deltanet

# Ensure third_party/cutlass submodule is initialized
!git submodule update --init --depth 1 third_party/cutlass || true

# Install build dependencies
!pip install -q ninja setuptools wheel einops

# Set MAX_JOBS=2 to prevent CPU/RAM exhaustion and kernel crashes on Google Colab
%env MAX_JOBS=2

# Compile and install DeltaNet v2 (mặc định tự biên dịch; dùng FORCE_REBUILD=1 nếu muốn ghi đè)
!pip install -v . --no-build-isolation

# Copy compiled .so into local deltanet_v2 folder to ensure in-tree imports work seamlessly
!cp build/lib.*/deltanet_v2/_C*.so deltanet_v2/ 2>/dev/null || true

## 3. (Tùy chọn) Đóng gói File Wheel (.whl)
Đóng gói các file đã biên dịch thành file `.whl` sạch (chỉ chứa `deltanet_v2/`, không chứa các file rác của container Colab) để tải về máy hoặc tái sử dụng.

In [ ]:
# ==============================================================================
# Đóng gói file Wheel (.whl) sạch chỉ chứa package deltanet_v2
# ==============================================================================
import os
import glob
import zipfile
from google.colab import files

%cd /content/Deltanet

# Dọn dẹp thư mục build cũ
!rm -rf dist build *.egg-info

# Đóng gói wheel từ gói deltanet_v2
!python setup.py bdist_wheel

# Kiểm tra nội dung bên trong wheel để đảm bảo không bị lẫn file ngoài của container Colab
wheel_files = glob.glob("dist/*.whl")
if wheel_files:
    target_whl = wheel_files[0]
    print("\n" + "=" * 60)
    print(f"DANH SÁCH FILE BÊN TRONG WHEEL: {os.path.basename(target_whl)}")
    print("=" * 60)
    with zipfile.ZipFile(target_whl, "r") as z:
        for member in z.namelist():
            print(f"  - {member}")
    print("=" * 60)

    # Tải file wheel về máy
    print(f"\n>> Đang chuẩn bị tải về: {os.path.basename(target_whl)}...")
    files.download(target_whl)
else:
    print("[!] Không tìm thấy file wheel trong /content/Deltanet/dist/!")

## 4. Install Baseline Libraries
Installs **Flash Linear Attention** (`fla`) and **NVIDIA GatedDeltaNet-2** (`NVlabs/GatedDeltaNet-2`).

In [ ]:
# Install Flash Linear Attention (FLA)
!pip install -q flash-linear-attention

# Install NVlabs GatedDeltaNet-2
!pip install -q git+https://github.com/nvlabs/gateddeltanet-2.git --no-deps || true

## 5. Numerical Correctness Sanity Test
Verifies numerical consistency between DeltaNet v2 CUDA kernels and the pure PyTorch reference.

In [ ]:
!python /content/Deltanet/tests/test_correctness.py

## 6. Benchmark: Autoregressive Inference (L = 1)
Measures single-step decode latency (ms/step), generation throughput (tokens/sec), and peak VRAM across batch sizes $B \in [1, 16, 64, 256, 1024, 2048]$.

In [ ]:
!python /content/Deltanet/tests/bench_inference.py --dim 768 --q-heads 8 --kv-heads 8 --batches 1,16,64,256,1024,2048

## 7. Benchmark: Training Pass (Forward + Backward)
Measures step time (ms), training throughput (tokens/sec), and peak VRAM across sequence lengths $L \in [512, 1024, 2048, 4096]$.

In [ ]:
!python /content/Deltanet/tests/bench_training.py --dim 768 --q-heads 8 --kv-heads 8 --batch-size 4 --seq-lens 512,1024,2048,4096

## 8. Run Full Benchmark Suite
Runs all benchmarks and prints consolidated comparison tables.

In [ ]:
!python /content/Deltanet/tests/bench_all.py